# Rotation Fit Probability — ANN vs Random Forest

Predicts the probability that an employee will succeed after a job rotation.

**Data sources**
- `Career_Transaction_Karyawan_2012-2026_excel.xlsx` — movement history
- `Dataset.xlsx` — employee attributes and performance ratings

**Pending inputs** (the notebook runs without them and skips the affected parts)
- Position → Job Family mapping for the 2024 and Jan-2025 org structures
- Gender and marital status from HRIS

**Target**: `success_fair` — did the employee outperform the typical 2025
outcome for people who started 2024 at the same rating level. This removes
the company-wide rating recalibration that otherwise dominates the label.


# Setup & Config

In [ ]:
# =============================================================
# §1  SETUP AND CONFIG
# =============================================================
import json
import os
import random
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CFG = {
    # ---------- sources ----------
    "career_path": "Career_Transaction_Karyawan_2012-2026_excel.xlsx",
    "career_skiprows": 4,
    "dataset_path": "Dataset.xlsx",
    "jobfamily_map_path": None,   # <- set when available: 2 cols, position + job_family
    "hris_demog_path": None,      # <- set when available: NIK + gender + marital
    "out_dir": "outputs",

    # ---------- §2b cohort ----------
    "exclude_status": ["Outsource"],
    "check_old_status": True,
    "keep_career_transition": ["Movement"],
    "exclude_types": [
        "Pelaksana Tugas", "Pelaksana Harian", "Pejabat Sementara",
        "Pengganti Sementara", "Promotion", "Demotion",
        "Perubahan Kontrak", "Employment Status Change",
    ],
    "window_start": "2024-10-01",
    "window_end": "2025-03-31",
    "real_change_rule": "or",          # "or" | "and" | None
    "baseline_population": "all",      # "all" | "cohort"
    "target": "success_fair",          # "success_fair" | "success_simple"

    # ---------- §3 EQS ----------
    "rating_order": {
        "Need Improvement": 1, "Satisfactory": 2, "Successful": 3,
        "Excellent": 4, "Outstanding": 5, "Top Performer": 5,
    },
    "eqs_kinerja_points": {1: 50, 2: 75, 3: 100, 4: 110, 5: 120},
    "tfidf_threshold": 0.15,           # relevance cutoff for pelatihan/sertifikasi
    "aspirasi_points": {
        "Aspirasi Individual": 20, "Aspirasi Jobholder": 25,
        "Aspirasi Unit": 25, "Aspirasi Supervisor": 30,
    },
    "aspirasi_match_threshold": 0.35,

    # ---------- §4 similarity ----------
    "similarity_dims": ["usia", "pendidikan_years", "job_family", "gender", "marital"],
    "min_team_size": 4,
    "team_fallback": ["destination_org_unit", "Group", "Job Fam"],

    # ---------- §5/§6 experiment grid ----------
    "split_ratios": [0.30, 0.25, 0.20],   # test size
    "seeds": [1, 2, 3, 4, 5],
    "scalers": ["standard", "minmax"],
    "ann_backend": "auto",                # "auto" | "keras" | "sklearn"
}

os.makedirs(CFG["out_dir"], exist_ok=True)

import sklearn
print("pandas", pd.__version__, "| numpy", np.__version__, "| sklearn", sklearn.__version__)
print("run started", datetime.now().isoformat(timespec="seconds"))


# Data Load

In [ ]:
# =============================================================
# §2  LOAD AND VALIDATE
# =============================================================
def _norm(s):
    return s.astype("string").str.strip().str.casefold()


def _key(s):
    """IDs are not purely numeric — e.g. '1486.020-R'."""
    return s.astype(str).str.strip()


car = pd.read_excel(CFG["career_path"], skiprows=CFG["career_skiprows"])
car.columns = car.columns.str.strip()
car["Start Date"] = pd.to_datetime(car["Start Date"], errors="coerce")

emp = pd.read_excel(CFG["dataset_path"])
emp.columns = emp.columns.str.strip()

REQUIRED_CAR = ["Employee No", "Start Date", "Career Transition", "Transition Type",
                "Employment Status", "Old Employment Status",
                "Old Position", "Position",
                "Old Organization Unit", "Organization Unit"]
REQUIRED_EMP = ["NIK", "Kinerja 2024", "Kinerja 2025", "Job Fit",
                "Tgl Lahir", "Level Pendidikan 1", "Divisi", "Group", "Job Fam"]

for col in REQUIRED_CAR:
    assert col in car.columns, f"career file missing column: {col}"
for col in REQUIRED_EMP:
    assert col in emp.columns, f"dataset missing column: {col}"

car["_k"] = _key(car["Employee No"])
emp["_k"] = _key(emp["NIK"])

# Employee Name is NOT unique — always key on the ID.
assert emp["_k"].is_unique, "NIK is not unique in Dataset.xlsx"

print(f"career rows {len(car):,} | employees {emp['_k'].nunique():,}")
print(f"names shared by 2+ employees: {(emp.groupby('Nama')['_k'].nunique() > 1).sum()}")


In [ ]:
# =============================================================
# §2b  COHORT CONSTRUCTION AND TARGET LABELLING
# =============================================================
def _excl_mask(types, terms):
    if not terms:
        return pd.Series(False, index=types.index)
    esc = [t.strip().casefold() for t in terms]
    return types.str.contains("|".join(esc), na=False, regex=True)


funnel = {"Employees in Dataset.xlsx": emp["_k"].nunique()}

excl = [s.strip().casefold() for s in CFG["exclude_status"]]
ok = ~_norm(car["Employment Status"]).isin(excl)
if CFG["check_old_status"]:
    ok &= ~_norm(car["Old Employment Status"]).isin(excl)
if CFG["keep_career_transition"]:
    ok &= _norm(car["Career Transition"]).isin(
        [f.strip().casefold() for f in CFG["keep_career_transition"]])
ok &= ~_excl_mask(_norm(car["Transition Type"]), CFG["exclude_types"])
ok &= car["_k"].isin(set(emp["_k"]))

mv = car[ok].copy()
funnel["Rotated at any time (valid movement)"] = mv["_k"].nunique()

win = mv[mv["Start Date"].between(CFG["window_start"], CFG["window_end"])].copy()
funnel[f"Rotated {CFG['window_start']} to {CFG['window_end']}"] = win["_k"].nunique()

# Anchor = LAST qualifying move inside the window. Later moves (the Sept 2025
# restructure) are contamination of the outcome, not the studied event.
anchor = win.sort_values("Start Date").groupby("_k", as_index=False).tail(1).copy()

anchor["position_changed"] = _norm(anchor["Old Position"]) != _norm(anchor["Position"])
anchor["org_unit_changed"] = (_norm(anchor["Old Organization Unit"])
                              != _norm(anchor["Organization Unit"]))
if CFG["real_change_rule"] == "or":
    anchor = anchor[anchor["position_changed"] | anchor["org_unit_changed"]]
elif CFG["real_change_rule"] == "and":
    anchor = anchor[anchor["position_changed"] & anchor["org_unit_changed"]]
funnel["Position or org unit actually changed"] = anchor["_k"].nunique()

anchor_slim = anchor[[
    "_k", "Start Date", "Transition Type", "Old Position", "Position",
    "Old Organization Unit", "Organization Unit", "Old Grade", "Grade",
    "position_changed", "org_unit_changed",
]].rename(columns={
    "Start Date": "rotation_date",
    "Old Position": "origin_position", "Position": "destination_position",
    "Old Organization Unit": "origin_org_unit",
    "Organization Unit": "destination_org_unit",
    "Old Grade": "origin_grade", "Grade": "destination_grade",
})

O = CFG["rating_order"]
emp["_o24"] = emp["Kinerja 2024"].map(O)
emp["_o25"] = emp["Kinerja 2025"].map(O)

df = emp.merge(anchor_slim, on="_k", how="inner")
df["k24_ord"] = df["Kinerja 2024"].map(O)
df["k25_ord"] = df["Kinerja 2025"].map(O)
df = df[df["k24_ord"].notna() & df["k25_ord"].notna()].copy()
funnel["Both 2024 and 2025 ratings present"] = len(df)

# --- label 1: naive ---
df["success_simple"] = (df["k25_ord"] >= df["k24_ord"]).astype(int)

# --- label 2: fair (recommended) ---
# Baseline = mean 2025 outcome among everyone who started 2024 at the same
# level. Success = beat that baseline. Compares like with like, so a top
# performer is not penalised for having started high.
if CFG["baseline_population"] == "all":
    expected = emp.dropna(subset=["_o24", "_o25"]).groupby("_o24")["_o25"].mean()
else:
    expected = df.groupby("k24_ord")["k25_ord"].mean()

df["expected_k25"] = df["k24_ord"].map(expected)
df["residual_k25"] = df["k25_ord"] - df["expected_k25"]
df["success_fair"] = (df["residual_k25"] > 0).astype(int)

# --- diagnostic only: within-year percentiles (for the thesis) ---
for yr, oc in [("2024", "_o24"), ("2025", "_o25")]:
    r = emp[oc].rank(pct=True) * 100
    lut = pd.Series(r.values, index=emp[oc].values).groupby(level=0).mean()
    df[f"kinerja_{yr}_pct"] = df[f"k{yr[-2:]}_ord"].map(lut).round(1)

cohort = df.reset_index(drop=True)

print("INCLUSION FUNNEL")
for k, v in funnel.items():
    print(f"  {k:<44} {v:>6,}")
print(f"\nFINAL COHORT: {len(cohort):,} employees "
      f"({cohort['rotation_date'].min().date()} to {cohort['rotation_date'].max().date()})")
for col in ["success_simple", "success_fair"]:
    p = int(cohort[col].sum())
    print(f"  {col:<15} success {p:>4,} ({p/len(cohort):.1%})  fail {len(cohort)-p:>4,}")

print("\nSUCCESS RATE BY 2024 RATING (ceiling-effect check)")
chk = (cohort.groupby("Kinerja 2024")
       .agg(n=("success_fair", "size"),
            simple=("success_simple", "mean"), fair=("success_fair", "mean")))
print((chk.assign(simple=lambda x: (x.simple * 100).round(1),
                  fair=lambda x: (x.fair * 100).round(1))).to_string())


# Define Scoring and Else

In [ ]:
# =============================================================
# §3  EQS SCORING
# =============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---- optional external inputs -------------------------------
JF_MAP = None
if CFG["jobfamily_map_path"]:
    _jf = pd.read_excel(CFG["jobfamily_map_path"])
    _jf.columns = [c.strip().lower() for c in _jf.columns]
    JF_MAP = dict(zip(_norm(_jf.iloc[:, 0]), _jf.iloc[:, 1]))
    print(f"job family mapping loaded: {len(JF_MAP):,} positions")
else:
    print("NOTE: no job family mapping — origin/destination job family will be blank,")
    print("      similarity will use demographic dimensions only.")

if CFG["hris_demog_path"]:
    _dg = pd.read_excel(CFG["hris_demog_path"])
    _dg.columns = [c.strip() for c in _dg.columns]
    _dg["_k"] = _key(_dg.iloc[:, 0])
    cohort = cohort.merge(_dg.drop(columns=[_dg.columns[0]]), on="_k", how="left")
else:
    print("NOTE: no HRIS demographics — gender and marital status unavailable.")

# ---- 1. Kinerja (0-120) — pre-rotation rating only ----------
cohort["eqs_kinerja"] = cohort["k24_ord"].map(CFG["eqs_kinerja_points"])

# ---- 2. Kesesuaian kompetensi (0-100) — Job Fit -------------
cohort["eqs_kompetensi"] = (cohort["Job Fit"] * 100).clip(0, 100)

# ---- 3. Rekam jejak (0 / 50 / 100) --------------------------
# 0   = currently serving a disciplinary sanction
# 50  = past record
# 100 = clean (blank means no record, not missing data)
_aktif = cohort["Catatan Disiplin (Aktif)"].notna()
_past = cohort["Catatan Disiplin (Non-aktif)"].notna()
cohort["eqs_rekam_jejak"] = np.where(_aktif, 0, np.where(_past, 50, 100))

# ---- 4/5. Pelatihan and Sertifikasi relevance (0 / 100) -----
# EQS defines these as supporting the TARGET position, so relevance is
# measured against destination_position text, not against a raw count.
PEL = [f"Pelatihan {i}" for i in range(1, 6)]
SER = [f"Sertifikasi {i}" for i in range(1, 4)]


def _blob(row, cols):
    parts = [str(row[c]) for c in cols if c in row.index and pd.notna(row[c])]
    return " ; ".join(parts).replace("\n", " ")


def relevance_flag(text_series, target_series, threshold):
    """1 when the employee's text is semantically close to the target role."""
    docs = list(text_series.fillna("")) + list(target_series.fillna(""))
    if not any(d.strip() for d in docs):
        return np.zeros(len(text_series), dtype=int)
    vec = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
    M = vec.fit_transform(docs)
    n = len(text_series)
    sims = cosine_similarity(M[:n], M[n:]).diagonal()
    out = (sims >= threshold).astype(int)
    out[text_series.fillna("").str.strip() == ""] = 0
    return out


cohort["_pel_text"] = cohort.apply(lambda r: _blob(r, PEL), axis=1)
cohort["_ser_text"] = cohort.apply(lambda r: _blob(r, SER), axis=1)

cohort["eqs_pelatihan"] = relevance_flag(
    cohort["_pel_text"], cohort["destination_position"], CFG["tfidf_threshold"]) * 100
cohort["eqs_sertifikasi"] = relevance_flag(
    cohort["_ser_text"], cohort["destination_position"], CFG["tfidf_threshold"]) * 100

# ---- 6. Aspirasi (0 / 20 / 25 / 30) -------------------------
# Score by WHICH column names the destination role. Supervisor outranks
# Unit and Jobholder, which outrank Individual — so take the max.
def aspirasi_score(row):
    target = str(row.get("destination_position", "")).strip().casefold()
    if not target:
        return 0
    best = 0
    for col, pts in CFG["aspirasi_points"].items():
        val = row.get(col)
        if pd.isna(val):
            continue
        for line in str(val).split("\n"):
            line = line.strip().lstrip("-").strip().casefold()
            if not line:
                continue
            a, b = set(line.split()), set(target.split())
            if not b:
                continue
            if len(a & b) / len(b) >= CFG["aspirasi_match_threshold"]:
                best = max(best, pts)
    return best


cohort["eqs_aspirasi"] = cohort.apply(aspirasi_score, axis=1)

# ---- 7. Pengalaman — DEFERRED -------------------------------
# Masa kerja pada job family tujuan. Needs the job family mapping plus
# position history from the career file. Left out of the feature set.
cohort["eqs_pengalaman"] = np.nan

EQS_COLS = ["eqs_kinerja", "eqs_kompetensi", "eqs_rekam_jejak",
            "eqs_pelatihan", "eqs_sertifikasi", "eqs_aspirasi"]

print("EQS DIMENSIONS")
print(cohort[EQS_COLS].describe().T[["count", "mean", "min", "max"]].round(2).to_string())
print("\nzero-variance check (a constant feature cannot help the model):")
for c in EQS_COLS:
    top = cohort[c].value_counts(normalize=True).iloc[0]
    flag = "  <-- near constant" if top > 0.95 else ""
    print(f"  {c:<18} most common value covers {top:.1%}{flag}")


# Calculate Similarity Score

In [ ]:
# =============================================================
# §4  SIMILARITY SCORE (M2)
# =============================================================
# "How similar is this employee to the team they are moving into."
# Gower distance handles mixed numeric/categorical data natively.
# Implemented directly so the notebook has no extra dependency.

# ---- demographic features -----------------------------------
cohort["usia"] = ((cohort["rotation_date"] - pd.to_datetime(cohort["Tgl Lahir"],
                                                            errors="coerce")).dt.days / 365.25)
emp["usia"] = ((pd.Timestamp(CFG["window_start"]) - pd.to_datetime(emp["Tgl Lahir"],
                                                                   errors="coerce")).dt.days / 365.25)

EDU_YEARS = {
    "sma": 12, "smu": 12, "smk": 12, "stm": 12, "slta": 12, "ma": 12, "paket c": 12,
    "d1": 13, "d2": 14, "d3": 15, "d4": 16, "s1": 16, "profesi": 17, "s2": 18, "s3": 21,
}


def edu_years(v):
    s = str(v).strip().casefold()
    for k in sorted(EDU_YEARS, key=len, reverse=True):
        if k in s:
            return EDU_YEARS[k]
    return np.nan


cohort["pendidikan_years"] = cohort["Level Pendidikan 1"].map(edu_years)
emp["pendidikan_years"] = emp["Level Pendidikan 1"].map(edu_years)

# ---- job family (needs the mapping) -------------------------
if JF_MAP:
    cohort["origin_job_family"] = _norm(cohort["origin_position"]).map(JF_MAP)
    cohort["destination_job_family"] = _norm(cohort["destination_position"]).map(JF_MAP)
    emp["job_family"] = _norm(emp["Position"]).map(JF_MAP).fillna(emp["Job Fam"])
else:
    cohort["origin_job_family"] = np.nan
    cohort["destination_job_family"] = np.nan
    emp["job_family"] = emp["Job Fam"]

cohort["job_family_changed"] = np.where(
    cohort[["origin_job_family", "destination_job_family"]].notna().all(axis=1),
    (cohort["origin_job_family"] != cohort["destination_job_family"]).astype(float),
    np.nan)

# ---- reconstruct team rosters as of the rotation date -------
# Dataset.xlsx holds the POST-September-2025 structure, so team membership
# must come from the career file instead.
roster_src = car[car["_k"].isin(set(emp["_k"]))].copy()


def roster_as_of(date):
    past = roster_src[roster_src["Start Date"] <= date]
    last = past.sort_values("Start Date").groupby("_k").tail(1)
    return dict(zip(last["_k"], _norm(last["Organization Unit"])))


# ---- Gower distance ------------------------------------------
def gower(a_num, B_num, ranges, a_cat, B_cat):
    """Mean per-feature distance from one row to many rows."""
    parts = []
    if a_num.size:
        d = np.abs(B_num - a_num) / np.where(ranges == 0, 1, ranges)
        parts.append(d)
    if a_cat.size:
        parts.append((B_cat != a_cat).astype(float))
    if not parts:
        return np.array([])
    D = np.hstack(parts)
    return np.nanmean(D, axis=1)


AVAILABLE = [d for d in CFG["similarity_dims"]
             if d in emp.columns or d in {"job_family"}]
NUM_DIMS = [d for d in AVAILABLE if d in {"usia", "pendidikan_years"}]
CAT_DIMS = [d for d in AVAILABLE if d in {"job_family", "gender", "marital"}
            and d in emp.columns]

print(f"similarity dimensions in use: {NUM_DIMS + CAT_DIMS}")
missing = [d for d in CFG["similarity_dims"] if d not in NUM_DIMS + CAT_DIMS]
if missing:
    print(f"  unavailable (pending input): {missing}")

emp_num = emp[NUM_DIMS].to_numpy(dtype=float) if NUM_DIMS else np.empty((len(emp), 0))
emp_cat = emp[CAT_DIMS].astype(str).to_numpy() if CAT_DIMS else np.empty((len(emp), 0))
ranges = np.nanmax(emp_num, axis=0) - np.nanmin(emp_num, axis=0) if NUM_DIMS else np.array([])
emp_idx = {k: i for i, k in enumerate(emp["_k"])}

sim_raw, sim_level = [], []
roster_cache = {}

for _, row in cohort.iterrows():
    d = row["rotation_date"]
    if d not in roster_cache:
        roster_cache[d] = roster_as_of(d)
    roster = roster_cache[d]

    target_unit = str(row["destination_org_unit"]).strip().casefold()
    members = [k for k, u in roster.items() if u == target_unit and k != row["_k"]]
    level = "destination_org_unit"

    if len(members) < CFG["min_team_size"]:          # fallback for tiny teams
        grp = emp.loc[emp["_k"] == row["_k"], "Group"]
        if len(grp):
            members = [k for k in emp.loc[emp["Group"] == grp.iloc[0], "_k"]
                       if k != row["_k"]]
            level = "Group"

    if len(members) < CFG["min_team_size"]:
        sim_raw.append(np.nan)
        sim_level.append("insufficient")
        continue

    i = emp_idx[row["_k"]]
    j = [emp_idx[k] for k in members if k in emp_idx]
    a_num = emp_num[i:i + 1] if NUM_DIMS else np.empty((1, 0))
    a_cat = emp_cat[i:i + 1] if CAT_DIMS else np.empty((1, 0))
    dist = gower(a_num, emp_num[j], ranges, a_cat, emp_cat[j])
    sim_raw.append(1 - np.nanmean(dist))
    sim_level.append(level)

cohort["_sim_raw"] = sim_raw
cohort["similarity_fallback_level"] = sim_level
# rescale to 1-100 across the cohort, as specified in Tabel 3
cohort["similarity_score"] = (cohort["_sim_raw"].rank(pct=True) * 99 + 1).round(1)

print(f"\nsimilarity computed for {cohort['similarity_score'].notna().sum():,}"
      f" of {len(cohort):,} employees")
print(cohort["similarity_fallback_level"].value_counts().to_string())


# Data Preprocessing

In [ ]:
# =============================================================
# §5  FEATURE TABLE, SPLITS AND PREPROCESSING PIPELINE
# =============================================================
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

NUMERIC_FEATURES = EQS_COLS + ["usia", "pendidikan_years", "similarity_score"]
CATEGORICAL_FEATURES = [c for c in ["Level Jabatan", "job_family_changed",
                                    "gender", "marital"] if c in cohort.columns]

NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if cohort[c].notna().any()]
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if cohort[c].notna().any()]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = CFG["target"]

X = cohort[FEATURES].copy()
y = cohort[TARGET].astype(int).to_numpy()

print(f"features ({len(FEATURES)}): {FEATURES}")
print(f"target: {TARGET} | positives {y.sum():,} of {len(y):,} ({y.mean():.1%})")


def make_preprocessor(kind):
    scaler = StandardScaler() if kind == "standard" else MinMaxScaler()
    num = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", scaler)])
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=5))])
    blocks = [("num", num, NUMERIC_FEATURES)]
    if CATEGORICAL_FEATURES:
        blocks.append(("cat", cat, CATEGORICAL_FEATURES))
    return ColumnTransformer(blocks)


# The scaler is fitted INSIDE the pipeline, so it only ever sees training
# folds. This is what prevents leakage — not a comment, a structure.
def split_grid():
    for test_size in CFG["split_ratios"]:
        for seed in CFG["seeds"]:
            sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size,
                                         random_state=seed)
            tr, te = next(sss.split(X, y))
            yield test_size, seed, tr, te


print(f"\nsplit grid: {len(CFG['split_ratios'])} ratios x {len(CFG['seeds'])} seeds "
      f"= {len(CFG['split_ratios']) * len(CFG['seeds'])} partitions")


# Model Development

In [ ]:
# =============================================================
# §6  MODEL TRAINING GRID
# =============================================================
import time

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score,
                             roc_auc_score)
from sklearn.neural_network import MLPClassifier

# --- ANN backend -------------------------------------------------
ANN_BACKEND = CFG["ann_backend"]
if ANN_BACKEND in ("auto", "keras"):
    try:
        import tensorflow as tf
        tf.random.set_seed(SEED)
        ANN_BACKEND = "keras"
    except Exception:
        if CFG["ann_backend"] == "keras":
            raise
        ANN_BACKEND = "sklearn"
print(f"ANN backend: {ANN_BACKEND}")


def build_keras_ann(n_features, seed):
    from tensorflow import keras
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([
        keras.layers.Input(shape=(n_features,)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
    return m


def fit_predict_ann(Xtr, ytr, Xte, seed):
    """Returns predicted probabilities for the positive class."""
    if ANN_BACKEND == "keras":
        from tensorflow import keras
        n_pos, n_neg = ytr.sum(), len(ytr) - ytr.sum()
        cw = {0: len(ytr) / (2 * n_neg), 1: len(ytr) / (2 * n_pos)}
        model = build_keras_ann(Xtr.shape[1], seed)
        es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                           restore_best_weights=True)
        model.fit(Xtr, ytr, validation_split=0.2, epochs=200, batch_size=32,
                  class_weight=cw, callbacks=[es], verbose=0)
        return model.predict(Xte, verbose=0).ravel(), model
    clf = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=600,
                        early_stopping=True, n_iter_no_change=15,
                        random_state=seed)
    clf.fit(Xtr, ytr)
    return clf.predict_proba(Xte)[:, 1], clf


def make_model(name, seed):
    if name == "dummy":
        return DummyClassifier(strategy="stratified", random_state=seed)
    if name == "logreg":
        return LogisticRegression(max_iter=2000, class_weight="balanced",
                                  random_state=seed)
    if name == "rf":
        return RandomForestClassifier(n_estimators=400, min_samples_leaf=3,
                                      class_weight="balanced_subsample",
                                      random_state=seed, n_jobs=-1)
    raise ValueError(name)


MODELS = ["dummy", "logreg", "rf", "ann"]
rows, fitted = [], {}

for scaler_kind in CFG["scalers"]:
    for test_size, seed, tr, te in split_grid():
        ratio = f"{int((1 - test_size) * 100)}:{int(test_size * 100)}"
        pre = make_preprocessor(scaler_kind)
        Xtr = pre.fit_transform(X.iloc[tr])
        Xte = pre.transform(X.iloc[te])
        Xtr = np.asarray(Xtr.todense()) if hasattr(Xtr, "todense") else Xtr
        Xte = np.asarray(Xte.todense()) if hasattr(Xte, "todense") else Xte
        ytr, yte = y[tr], y[te]

        for name in MODELS:
            t0 = time.time()
            if name == "ann":
                proba, est = fit_predict_ann(Xtr, ytr, Xte, seed)
            else:
                est = make_model(name, seed)
                est.fit(Xtr, ytr)
                proba = est.predict_proba(Xte)[:, 1]
            elapsed = time.time() - t0
            pred = (proba >= 0.5).astype(int)

            rows.append({
                "model": name, "scaler": scaler_kind, "ratio": ratio, "seed": seed,
                "auc": roc_auc_score(yte, proba),
                "pr_auc": average_precision_score(yte, proba),
                "f1": f1_score(yte, pred, zero_division=0),
                "brier": brier_score_loss(yte, proba),
                "fit_time": elapsed,
            })
            fitted[(name, scaler_kind, ratio, seed)] = (pre, est)

results = pd.DataFrame(rows)
print(f"\ncompleted {len(results):,} fits")
print(results.groupby("model")[["auc", "pr_auc", "f1", "brier", "fit_time"]]
      .mean().round(3).to_string())


# Model Evaluation

In [ ]:
# =============================================================
# §7  EVALUATION
# =============================================================
import matplotlib.pyplot as plt

summary = (results.groupby(["model", "scaler", "ratio"])
           .agg(auc_mean=("auc", "mean"), auc_std=("auc", "std"),
                pr_mean=("pr_auc", "mean"), brier_mean=("brier", "mean"))
           .round(3).reset_index())
print("MEAN +/- SD ACROSS SEEDS")
print(summary.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) AUC by model and split ratio, with seed variability as error bars
piv = results.pivot_table(index="ratio", columns="model", values="auc", aggfunc="mean")
err = results.pivot_table(index="ratio", columns="model", values="auc", aggfunc="std")
piv.plot(kind="bar", yerr=err, ax=axes[0], capsize=3, rot=0)
axes[0].set_title("ROC-AUC by split ratio\n(error bars = SD across seeds)")
axes[0].set_ylabel("ROC-AUC")
axes[0].axhline(0.5, ls="--", c="grey", lw=1)

# (b) standardisation vs normalisation
piv2 = results.pivot_table(index="model", columns="scaler", values="auc", aggfunc="mean")
piv2.plot(kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Standardised vs normalised\n(trees are scale-invariant)")
axes[1].set_ylabel("ROC-AUC")

# (c) calibration — the probabilities are the deliverable, so they must mean something
best = summary.sort_values("pr_mean", ascending=False).iloc[0]
key = (best["model"], best["scaler"], best["ratio"], CFG["seeds"][0])
pre, est = fitted[key]
_, _, tr, te = next(s for s in split_grid()
                    if f"{int((1 - s[0]) * 100)}:{int(s[0] * 100)}" == best["ratio"]
                    and s[1] == CFG["seeds"][0])
Xte_b = pre.transform(X.iloc[te])
Xte_b = np.asarray(Xte_b.todense()) if hasattr(Xte_b, "todense") else Xte_b
proba_b = (est.predict(Xte_b, verbose=0).ravel()
           if best["model"] == "ann" and ANN_BACKEND == "keras"
           else est.predict_proba(Xte_b)[:, 1])

from sklearn.calibration import calibration_curve
frac, mean_pred = calibration_curve(y[te], proba_b, n_bins=8, strategy="quantile")
axes[2].plot([0, 1], [0, 1], ls="--", c="grey", lw=1)
axes[2].plot(mean_pred, frac, marker="o")
axes[2].set_title(f"Calibration — {best['model']}\n(on the diagonal = trustworthy %)")
axes[2].set_xlabel("predicted probability")
axes[2].set_ylabel("observed frequency")

plt.tight_layout()
plt.savefig(f"{CFG['out_dir']}/evaluation.png", dpi=140)
plt.show()

print(f"\nBEST BY PR-AUC: {best['model']} / {best['scaler']} / {best['ratio']}"
      f"  PR-AUC {best['pr_mean']:.3f}  AUC {best['auc_mean']:.3f}"
      f" +/- {best['auc_std']:.3f}")
print(f"baseline PR-AUC (always predict positive rate) = {y.mean():.3f}")


# Insight Extraction

In [ ]:
# =============================================================
# §8  INTERPRETATION
# =============================================================
from sklearn.inspection import permutation_importance

rf_key = [k for k in fitted if k[0] == "rf"][0]
pre_rf, rf = fitted[rf_key]
_, _, tr_rf, te_rf = next(iter(split_grid()))
Xte_rf = pre_rf.transform(X.iloc[te_rf])
Xte_rf = np.asarray(Xte_rf.todense()) if hasattr(Xte_rf, "todense") else Xte_rf

try:
    names = list(pre_rf.get_feature_names_out())
except Exception:
    names = [f"f{i}" for i in range(Xte_rf.shape[1])]

# Permutation importance, not impurity importance — the latter is biased
# toward high-cardinality features, and this dataset has several.
pi = permutation_importance(rf, Xte_rf, y[te_rf], n_repeats=30,
                            random_state=SEED, scoring="average_precision")
imp = (pd.DataFrame({"feature": names, "mean": pi.importances_mean,
                     "std": pi.importances_std})
       .sort_values("mean", ascending=False))
print("PERMUTATION IMPORTANCE (top 12)")
print(imp.head(12).round(4).to_string(index=False))
print("\nWith ~144 positives, importances are unstable — read the SD column,")
print("and treat any feature whose SD exceeds its mean as unranked.")

fig, ax = plt.subplots(figsize=(7, 5))
top = imp.head(12).iloc[::-1]
ax.barh(top["feature"], top["mean"], xerr=top["std"])
ax.set_title("What drives the prediction")
plt.tight_layout()
plt.savefig(f"{CFG['out_dir']}/importance.png", dpi=140)
plt.show()

# Subgroup check — materially different accuracy across groups is a finding
# that needs addressing before this influences a real decision.
sub = cohort.iloc[te_rf].copy()
sub["_proba"] = rf.predict_proba(Xte_rf)[:, 1]
sub["_y"] = y[te_rf]
for col in [c for c in ["gender", "Level Jabatan"] if c in sub.columns]:
    print(f"\nAUC by {col}:")
    for g, grp in sub.groupby(col):
        if grp["_y"].nunique() > 1 and len(grp) >= 20:
            print(f"  {str(g)[:28]:<30} n={len(grp):>4}"
                  f"  AUC={roc_auc_score(grp['_y'], grp['_proba']):.3f}")


# Save Model

In [ ]:
# =============================================================
# §9  SAVE AND SCORE NEW DATA
# =============================================================
import joblib

best_key = [k for k in fitted if k[0] == best["model"] and k[1] == best["scaler"]][0]
pre_best, est_best = fitted[best_key]

# Calibrate — Random Forest probabilities are typically overconfident, and
# the percentage is the actual deliverable.
from sklearn.calibration import CalibratedClassifierCV

_, _, tr_b, te_b = next(iter(split_grid()))
Xtr_b = pre_best.fit_transform(X.iloc[tr_b])
Xtr_b = np.asarray(Xtr_b.todense()) if hasattr(Xtr_b, "todense") else Xtr_b

if best["model"] in {"rf", "logreg"}:
    calibrated = CalibratedClassifierCV(make_model(best["model"], SEED),
                                        method="isotonic", cv=5)
    calibrated.fit(Xtr_b, y[tr_b])
    final_model = calibrated
else:
    final_model = est_best

artifact = {"preprocessor": pre_best, "model": final_model,
            "features": FEATURES,
            "numeric_features": NUMERIC_FEATURES,
            "categorical_features": CATEGORICAL_FEATURES}
joblib.dump(artifact, f"{CFG['out_dir']}/rotation_fit_model.joblib")

metadata = {
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "target": TARGET,
    "cohort_size": int(len(cohort)),
    "positive_rate": float(y.mean()),
    "window": [CFG["window_start"], CFG["window_end"]],
    "features": FEATURES,
    "best_config": {k: (v if not isinstance(v, np.generic) else v.item())
                    for k, v in best.to_dict().items()},
    "pending_inputs": {"job_family_map": JF_MAP is not None,
                       "hris_demographics": CFG["hris_demog_path"] is not None},
    "known_limitations": [
        "2024 was the company's first rating cycle and was lenient; 2025 tightened.",
        "The Sept 2025 restructure affects the final quarter of the 2025 rating.",
        "eqs_pengalaman (masa kerja pada job family) not yet implemented.",
    ],
    "library_versions": {"pandas": pd.__version__, "numpy": np.__version__,
                         "sklearn": sklearn.__version__},
}
with open(f"{CFG['out_dir']}/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"saved model and metadata to {CFG['out_dir']}/")


def score_new(df_new, artifact_path=f"{CFG['out_dir']}/rotation_fit_model.joblib"):
    """Score unseen employee-destination pairs. Returns success probability in %.

    df_new must already carry the engineered columns (EQS dimensions,
    demographics, similarity_score) produced by sections 3 and 4.
    """
    art = joblib.load(artifact_path)
    missing = [c for c in art["features"] if c not in df_new.columns]
    if missing:
        raise ValueError(f"new data is missing required columns: {missing}")
    Z = art["preprocessor"].transform(df_new[art["features"]])
    Z = np.asarray(Z.todense()) if hasattr(Z, "todense") else Z
    p = (art["model"].predict(Z, verbose=0).ravel()
         if hasattr(art["model"], "predict_step")
         else art["model"].predict_proba(Z)[:, 1])

    # Out-of-distribution flag: a confident number for an unseen kind of
    # pairing is worse than no number at all.
    from sklearn.neighbors import NearestNeighbors
    ref = art["preprocessor"].transform(X)
    ref = np.asarray(ref.todense()) if hasattr(ref, "todense") else ref
    nn = NearestNeighbors(n_neighbors=5).fit(ref)
    d_new = nn.kneighbors(Z)[0].mean(axis=1)
    d_ref = nn.kneighbors(ref)[0].mean(axis=1)
    return pd.DataFrame({
        "rotation_success_pct": (p * 100).round(1),
        "out_of_distribution": d_new > np.percentile(d_ref, 95),
    }, index=df_new.index)


demo = score_new(cohort.head(5))
print("\nexample output:")
print(demo.to_string())

cohort.to_excel(f"{CFG['out_dir']}/cohort_final.xlsx", index=False)
results.to_csv(f"{CFG['out_dir']}/results_grid.csv", index=False)
print(f"\nwrote cohort_final.xlsx ({len(cohort):,} rows) and results_grid.csv")
